In [19]:
import torch
import numpy as np

### GPU ile çalışmak

In [20]:
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
print("Device Name:",torch.cuda.get_device_name())

CUDA: True
Device: cuda:0
Device Name: Tesla T4


In [21]:
# Cihaz seçimi
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [22]:
if torch.cuda.is_available():
    device = "cuda"
# macbook için
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [23]:
# tensor'larımızı seçili cihaz içine nasıl alabiliriz
my_tensor = torch.tensor([1,2,3])
my_tensor

tensor([1, 2, 3], device='cuda:0')

In [24]:
my_tensor.device

device(type='cuda', index=0)

In [25]:
# 1-) manuel
tensor_on_gpu = my_tensor.to(device)
tensor_on_gpu

tensor([1, 2, 3], device='cuda:0')

In [26]:
# my_tensor.numpy() ile tensor'ümüzü bir numpy array'e çevirebiliyoruz -> array([1,2,3])
# tensor_on_gpu.numpy() yapamayız ama, çünkü numpy cpu üzerinde çalışmaktadır
# tensor_on_gpu.cpu().numpy() ile yapılabilir ama
# PyTorch ile numpy arasındaki en büyük farklardan biri de gpu kullanımıdır zaten

In [27]:
# 2-) context manager
with torch.device(device):
  tensor2 = torch.tensor([1,2,3])
  layer = torch.nn.Linear(20,30)

In [28]:
tensor2.device

device(type='cuda', index=0)

In [29]:
layer.weight.device

device(type='cuda', index=0)

In [30]:
tensor3 = torch.tensor([1,2,3])
tensor3.device

device(type='cuda', index=0)

In [31]:
# 3-) torch.set_default_device
device

'cuda'

In [32]:
torch.set_default_device(device)

In [33]:
tensor4 = torch.tensor([1,2,3])
tensor4.device

device(type='cuda', index=0)

In [34]:
layer2 = torch.nn.Linear(10,40)
layer2.weight.device

device(type='cuda', index=0)

### Manipulating Tensor: reshaping, stacking, squeezing, unsqueezing

In [35]:
x = torch.arange(1,10,1)

In [36]:
print(x)
print(x.shape)
print(x.ndim)

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')
torch.Size([9])
1


In [37]:
x_reshaped = x.reshape(1,9)
print(x_reshaped)
print(x_reshaped.shape)
print(x_reshaped.ndim)

tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9]], device='cuda:0')
torch.Size([1, 9])
2


In [38]:
x_reshaped = x.reshape(3,3)
print(x_reshaped)
print(x_reshaped.shape)
print(x_reshaped.ndim)

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]], device='cuda:0')
torch.Size([3, 3])
2


In [39]:
x_reshaped = x.reshape(9,1)
print(x_reshaped)
print(x_reshaped.shape)
print(x_reshaped.ndim)

print()
x_reshaped[0] = 20
print(x_reshaped)

tensor([[1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7],
        [8],
        [9]], device='cuda:0')
torch.Size([9, 1])
2

tensor([[20],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9]], device='cuda:0')


In [40]:
# view
x_view = x.view(9,1)
print(x_view)
print(x_view.shape)
print(x_view.ndim)

print()
x_view[0] = 10
print(x_view)

tensor([[20],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9]], device='cuda:0')
torch.Size([9, 1])
2

tensor([[10],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9]], device='cuda:0')


## 🧩 Contiguous Tensors — `view` vs `reshape`

PyTorch'ta tensorlar bellekte **sıralı (contiguous)** ya da **dağınık (non-contiguous)**
olabilir.

> `.T` veya `permute()` gibi işlemler tensor'ın **bellek sırasını bozmadan**
> sadece görünümünü değiştirir → tensor artık **non-contiguous** olur.

---

### Fark Nedir?

| | `view()` | `reshape()` |
|---|---|---|
| Contiguous tensor'da | ✅ Çalışır | ✅ Çalışır |
| Non-contiguous tensor'da | ❌ **Hata verir** | ✅ Otomatik kopyalar, çalışır |
| Bellek kopyası oluşturur mu? | ❌ Hayır (aynı bellek) | ⚠️ Gerekirse evet |

---

### Özet
```python
X = torch.randn(3, 4)
X_T = X.T                      # Non-contiguous!

X_T.is_contiguous()            # False

X_T.view(12)                   # ❌ RuntimeError
X_T.reshape(12)                # ✅ Çalışır

X_T.contiguous().view(12)      # ✅ Önce contiguous yap, sonra view
```

> 💡 **Kural:** Emin değilsen **`reshape()`** kullan.  
> Performans kritikse ve tensor'ın contiguous olduğunu biliyorsan **`view()`** daha hızlıdır.

In [41]:
# https://medium.com/analytics-vidhya/pytorch-contiguous-vs-non-contiguous-tensor-view-understanding-view-reshape-73e10cdfa0dd

In [42]:
x = torch.tensor([[1,2,3],
                  [4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')

In [43]:
x.is_contiguous() # hafızada arka arkaya saklanıyor mu ? Evetse reshape ile de view ile de yapabiliriz

True

In [44]:
# Arka arkaya saklanmadığı durumlarda örneğin Transpose işleminde
y = x.t() # ya da y = x.T -> satırları sütuna çevirdik
y

tensor([[1, 4],
        [2, 5],
        [3, 6]], device='cuda:0')

In [45]:
y.is_contiguous() # hafızada arka arkaya mı saklanıyor ? Hayırsa sadece reshape uygulanabilir.

False

In [49]:
y.view(1,6)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [50]:
x.view(1,6)

tensor([[1, 2, 3, 4, 5, 6]], device='cuda:0')

In [51]:
# contiguous -> False view çalışmaz, reshape çalışır ama
y.reshape(1,6)

tensor([[1, 4, 2, 5, 3, 6]], device='cuda:0')

In [52]:
# Stacking

In [53]:
x

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')

In [54]:
torch.stack([x,x,x], dim=0)

tensor([[[1, 2, 3],
         [4, 5, 6]],

        [[1, 2, 3],
         [4, 5, 6]],

        [[1, 2, 3],
         [4, 5, 6]]], device='cuda:0')

In [55]:
torch.stack([x,x,x], dim=1)

tensor([[[1, 2, 3],
         [1, 2, 3],
         [1, 2, 3]],

        [[4, 5, 6],
         [4, 5, 6],
         [4, 5, 6]]], device='cuda:0')

In [56]:
# squeeze & unsqueeze

In [57]:
print(x)
print(x.shape)
print(x.dim())

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')
torch.Size([2, 3])
2


In [58]:
print(x.squeeze())
print(x.squeeze().shape)
print(x.squeeze().ndim)

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')
torch.Size([2, 3])
2


In [59]:
# bir değişiklik olmadı -> squeeze eğer yapabiliyorsa boyutu azaltabiliyor
z = torch.tensor([[1,2,3]])
print(z)
print(z.shape)
print(z.dim())

tensor([[1, 2, 3]], device='cuda:0')
torch.Size([1, 3])
2


In [60]:
print(z.squeeze())
print(z.squeeze().shape)
print(z.squeeze().ndim)

tensor([1, 2, 3], device='cuda:0')
torch.Size([3])
1


In [61]:
# unsqueeze

y = torch.tensor([[1,2,3],[4,5,6]])
print(y)
print(y.shape)
print(y.dim())

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')
torch.Size([2, 3])
2


In [62]:
print(y.unsqueeze(dim=0))
print(y.unsqueeze(dim=0).shape)
print(y.unsqueeze(dim=0).ndim)

tensor([[[1, 2, 3],
         [4, 5, 6]]], device='cuda:0')
torch.Size([1, 2, 3])
3


In [63]:
t = y.squeeze()
print(t)
print(t.shape)
print(t.ndim)

tensor([[1, 2, 3],
        [4, 5, 6]], device='cuda:0')
torch.Size([2, 3])
2


In [64]:
# permute

In [65]:
x = torch.rand(size=(224,224,3))
print(x)
print(x.shape)
print(x.ndim)

tensor([[[9.8772e-01, 1.2890e-01, 5.6210e-01],
         [5.2211e-01, 7.4448e-01, 5.9548e-01],
         [9.6471e-01, 8.9792e-01, 7.7297e-01],
         ...,
         [4.0627e-01, 6.1713e-01, 1.5666e-01],
         [6.4663e-01, 3.9677e-01, 2.2560e-02],
         [6.3400e-01, 9.0377e-01, 2.9105e-01]],

        [[7.4879e-01, 2.2626e-01, 6.5603e-01],
         [9.9354e-01, 6.7367e-01, 7.7268e-01],
         [5.1181e-01, 9.8832e-01, 1.0220e-01],
         ...,
         [2.8815e-01, 3.7411e-01, 3.3408e-01],
         [6.3389e-01, 7.7298e-02, 5.3287e-01],
         [7.2138e-01, 6.3562e-01, 8.2745e-01]],

        [[8.7795e-01, 7.2060e-01, 9.8917e-01],
         [9.3309e-02, 9.0652e-01, 5.4714e-01],
         [4.5454e-01, 9.2730e-01, 6.0600e-01],
         ...,
         [1.7435e-01, 4.7878e-01, 3.5396e-01],
         [4.5107e-01, 2.9884e-02, 2.8056e-01],
         [8.9675e-01, 8.4811e-01, 5.0060e-01]],

        ...,

        [[1.5899e-01, 6.3298e-01, 4.4647e-01],
         [4.7405e-01, 4.7922e-01, 1.9864e-01]

In [66]:
x_permuted = x.permute(2, 0, 1) # axis 0->1, 1->2, 2->0

In [67]:
x_permuted.shape

torch.Size([3, 224, 224])

In [68]:
# indexing & scling

In [69]:
a = torch.arange(1,10,1)
a

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')

In [70]:
# Tensor'u (1,3,3) boyutuna getirir
# yani: 1 tane "katman", her katmanda 3x3'lük matris
# çıktı şekli: [[[1,2,3],
#                [4,5,6],
#                [7,8,9]]]
a = a.reshape(1,3,3)

In [71]:
# 0. indeksteki katmanı alır
# çünkü ilk boyut 1 olduğu için sadece 0 var
# sonuç: 3x3'lük matris
a[0]

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]], device='cuda:0')

In [72]:
# önce 0. katmanı alır -> 3x3 matris
# sonra onun 0. satırını alır
a[0][0]

tensor([1, 2, 3], device='cuda:0')

In [73]:
# önce katman (0), sonra satır (0), sonra sütun (0)
# yani ilk elemanı alır
a[0][0][0]

tensor(1, device='cuda:0')

In [74]:
# ":" -> tüm katmanları al (burada sadece 1 tane var)
# 0 -> her katmandaki 0. satırı al
a[:,0]

tensor([[1, 2, 3]], device='cuda:0')

In [75]:
# ":" -> tüm katmanlar
# 0 -> 0. satır
# 0 -> o satırdaki 0. sütun
a[:,0,0]

tensor([1], device='cuda:0')

In [76]:
# ":" -> tüm katmanlar
# 1 -> 1. satır (yani [4,5,6])
# 0 -> o satırın ilk elemanı
a[:,1,0]

tensor([4], device='cuda:0')

In [77]:
# ":" -> tüm katmanlar
# ":" -> tüm satırlar
# 0 -> her satırın 0. sütunu
# yani ilk sütunu alır
a[:, :, 0]

tensor([[1, 4, 7]], device='cuda:0')

In [78]:
# ":" -> tüm katmanlar
# 0 -> 0. satır
# ":" -> o satırdaki tüm sütunlar
a[:, 0, :]

tensor([[1, 2, 3]], device='cuda:0')

In [79]:
# random seed

In [80]:
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random_tensor = torch.rand(3,4)
random_tensor

tensor([[0.6130, 0.0101, 0.3984, 0.0403],
        [0.1563, 0.4825, 0.7362, 0.4060],
        [0.5189, 0.2867, 0.2416, 0.9228]], device='cuda:0')

---

## ✅ Neler Öğrendik?

---

### 1. 🔄 `reshape()` — Boyut Değiştirme

Bir tensorun eleman sayısını koruyarak farklı bir şekle dönüştürmeyi öğrendik.
Toplam eleman sayısı değişmediği sürece istediğimiz boyuta geçebiliriz.

> ⚠️ Kural: Yeni boyutların çarpımı = eski eleman sayısı olmalıdır.  
> Örnek: `12 eleman` → `(3,4)` ✅ → `(3,5)` ❌

---

### 2. 👁️ `view()` vs `reshape()` — Bellek Farkı

Her ikisi de boyut değiştirme işlemi yapar ancak bellekte farklı davranırlar.

| | `view()` | `reshape()` |
|---|---|---|
| Çalışma koşulu | Sadece **contiguous** tensor'larda | Her durumda çalışır |
| Bellek kopyası | ❌ Oluşturmaz (aynı belleği paylaşır) | ⚠️ Gerekirse kopyalar |
| Yan etki | Değişiklik orijinali de etkiler 🔁 | Kopya alırsa orijinal korunur |

> 💡 `view()` ile yapılan değişiklik orijinal tensoru da etkiler çünkü  
> ikisi aynı bellek alanını paylaşır.

---

### 3. 🧩 Contiguous Tensors

Tensor elemanları bellekte **arka arkaya** sıralanmışsa `contiguous` kabul edilir.
`.T`, `permute()` gibi işlemler bu sırayı bozarak tensoru **non-contiguous** yapar.
```python
# Contiguous kontrolü
tensor.is_contiguous()          # True / False

# Non-contiguous tensor'da çözüm
tensor.contiguous().view(...)   # Önce belleği düzelt, sonra view uygula
```

> 💡 `view()` non-contiguous tensor'da `RuntimeError` fırlatır.  
> `reshape()` ise bu durumu otomatik halleder.

---

### 4. 📚 `torch.stack()` — Tensor Yığınlama

Aynı boyuttaki tensorları **yeni bir eksen ekleyerek** birleştirir.
`dim` parametresi yeni boyutun nereye ekleneceğini belirler.

> 💡 `dim=0` → satır ekseninde yığınlar, `dim=1` → sütun ekseninde yığınlar.  
> Her iki durumda da çıktı boyutu bir artış gösterir.

---

### 5. 🗜️ `squeeze()` ve `unsqueeze()` — Boyut Ekleme / Çıkarma
```python
squeeze()          # Boyutu 1 olan ekseni kaldırır  [1,3] → [3]
unsqueeze(dim=N)   # N. konuma yeni bir eksen ekler  [3] → [1,3]
```

> ⚠️ `squeeze()` yalnızca boyutu **1 olan** eksenleri kaldırır,  
> diğer eksenlere dokunmaz.

---

### 6. 🔀 `permute()` — Eksenleri Yeniden Sırala

Boyutların sırasını yeniden düzenler, veriyi **kopyalamaz**.
Özellikle görüntü verisi formatları arasında geçişte kullanılır.
```python
# Görüntü formatları arasında geçiş
# PIL/NumPy formatı : (Yükseklik, Genişlik, Kanal)
# PyTorch formatı   : (Kanal, Yükseklik, Genişlik)

goruntu.permute(2, 0, 1)   # H,W,C  →  C,H,W
```

> ⚠️ `permute()` sonrası tensor **non-contiguous** hale gelir.

---

### 7. 🔍 Indexing & Slicing — Tensor Dilimleme

`[katman, satır, sütun]` mantığıyla çalışır. `:` o boyuttaki tüm elemanları seçer.
```python
a[0]           # 0. katmanı al
a[0][1]        # 0. katman → 1. satır
a[:, 1, :]     # tüm katmanlarda 1. satırın tüm sütunları
a[:, :, 2]     # tüm katman ve satırlarda yalnızca 2. sütun
```

---

### 8. 🎲 Random Seed — Tekrarlanabilirlik

Rastgele üretilen tensorların her çalıştırmada aynı değerleri vermesini sağlar.
Deneylerin **tekrarlanabilir** olması için standart bir pratiktir.
```python
torch.manual_seed(42)   # Seed'i sabitle → sonuçlar her seferinde aynı
```

---

### 📊 Genel Özet Tablosu

| İşlem | Metod | Ne Yapar? |
|---|---|---|
| Boyut değiştir | `reshape()` | Her tensor'da çalışır, gerekirse kopyalar |
| Boyut değiştir | `view()` | Sadece contiguous tensor'da, bellek paylaşır |
| Yığınla | `stack()` | Yeni boyut ekleyerek tensoru birleştirir |
| Boyut sil | `squeeze()` | Boyutu 1 olan ekseni kaldırır |
| Boyut ekle | `unsqueeze()` | Seçilen konuma yeni eksen ekler |
| Eksen sırala | `permute()` | Boyut sırasını yeniden düzenler |
| Dilimleme | `[:, 0, :]` | Belirli satır / sütun / katman seçer |
| Tekrarlanabilirlik | `manual_seed()` | Rastgele sonuçları sabitler |